In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -U ortools

In [ ]:
from ortools.sat.python import cp_model
import collections

def solve_luxury_set_optimization():
    model = cp_model.CpModel()

    # Data: (MachineID, Time)
    # M1=0, M2=1, M3=2
    jobs_data = [
        [(1, 4), (2, 3)],           # Job 1: Dress (M2, M3)
        [(0, 6), (1, 4), (2, 3)],   # Job 2: Handbag (M1, M2, M3)
        [(1, 3), (0, 4), (2, 3)]    # Job 3: Belt (M2, M1, M3)
    ]

    machines_count = 3
    all_machines = range(machines_count)
    
    # Calculate a safe upper bound for time (sum of all durations)
    horizon = sum(task[1] for job in jobs_data for task in job)

    # Named tuples for tasks
    task_type = collections.namedtuple('task_type', 'start end interval')
    all_tasks = {}
    machine_to_intervals = collections.defaultdict(list)

    for job_id, job in enumerate(jobs_data):
        last_task_end = 0
        for task_id, (machine, duration) in enumerate(job):
            suffix = f'_{job_id}_{task_id}'
            start_var = model.NewIntVar(0, horizon, 'start' + suffix)
            end_var = model.NewIntVar(0, horizon, 'end' + suffix)
            interval_var = model.NewIntervalVar(start_var, duration, end_var, 'interval' + suffix)

            # Constraint: A job's tasks must be sequential
            model.Add(start_var >= last_task_end)
            
            all_tasks[job_id, task_id] = task_type(start=start_var, end=end_var, interval=interval_var)
            machine_to_intervals[machine].append(interval_var)
            last_task_end = end_var

    # Constraint: No two jobs can use the same machine at the same time
    for machine in all_machines:
        model.AddNoOverlap(machine_to_intervals[machine])

    # Objective: Minimize Makespan
    obj_var = model.NewIntVar(0, horizon, 'makespan')
    model.AddMaxEquality(obj_var, [all_tasks[job_id, len(job)-1].end for job_id, job in enumerate(jobs_data)])
    model.Minimize(obj_var)

    # Solve
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        total_time = solver.ObjectiveValue()
        print(f"--- Optimization Results ---")
        print(f"Time to complete 1 full set: {total_time} minutes")
        print(f"Productivity: {60 / total_time:.2f} sets per hour (per assembly line)")
        
        # Output detailed schedule for the warehouse floor
        for j in range(len(jobs_data)):
            for t in range(len(jobs_data[j])):
                start = solver.Value(all_tasks[j, t].start)
                end = solver.Value(all_tasks[j, t].end)
                print(f"Job {j+1} Task {t+1} (M{jobs_data[j][t][0]+1}): {start} to {end} min")
    else:
        print("No solution found.")

solve_luxury_set_optimization()